In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel.csv')

In [3]:
batch_size = min(len(X), len(S), len(y))
X_tensor = torch.tensor(X.values[:batch_size], dtype=torch.float32).to(device='cuda')
S_tensor = torch.tensor(S.values[:batch_size], dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values[:batch_size, 0], dtype=torch.float32).to(device='cuda')

In [4]:
print(len(X_tensor))
print(len(S_tensor))
print(len(y_tensor))

13038
13038
13038


In [5]:
print(X_tensor[0])
print(X_tensor[0].size())

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0')
torch.Size([12288])


In [6]:
print(S_tensor[0])
print(S_tensor[0].size())

tensor([59.], device='cuda:0')
torch.Size([1])


In [7]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor(0.0053, device='cuda:0')
torch.Size([])


In [8]:
# tensor([[0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')
# torch.Size([1, 12288])

# tensor([[117]], device='cuda:0')
# torch.Size([1, 1])

# tensor([[-0.2253]], device='cuda:0', grad_fn=<TanhBackward0>)
# torch.Size([1, 1])

In [9]:
class LSTMNet(nn.Module):
    def __init__(self, input_size=12288, hidden_size=128, num_layers=2, speed_size=1, output_size=1):
        super(LSTMNet, self).__init__()
        self.device = 'cuda'
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True).to(self.device)
        self.fc = nn.Linear(hidden_size + speed_size, output_size).to(self.device)
        self.tanh = nn.Tanh().to(self.device)

    def forward(self, image, speed):
        image = image.to(self.device)
        speed = speed.to(self.device)
        batch_size = image.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)

        out, _ = self.lstm(image.unsqueeze(1), (h0, c0))
        out = out[:, -1, :]

        out = torch.cat((out, speed), dim=1).to(self.device)

        out = self.fc(out).to(self.device)
        output = self.tanh(out).to(self.device)
        return output

In [10]:
criterion = nn.MSELoss()
model = LSTMNet().cuda()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Создание DataLoader
dataset = TensorDataset(X_tensor, S_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# Цикл обучения
num_epochs = 100
for epoch in range(num_epochs):
    total_loss = 0
    for inputs_image, inputs_speed, targets in dataloader:
        # Обнуление градиентов
        optimizer.zero_grad()
        
        # Прямой проход (forward pass)
        outputs = model(inputs_image, inputs_speed)
        
        # Вычисление функции потерь
        loss = criterion(outputs.squeeze(), targets)
        
        # Обратное распространение (backward pass) и оптимизация
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Вывод среднего значения функции потерь после каждой эпохи
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader)}")

Epoch 1, Loss: 0.9228515028953552
Epoch 2, Loss: 0.9018556475639343
Epoch 3, Loss: 0.8729455471038818
Epoch 4, Loss: 0.8310749530792236
Epoch 5, Loss: 0.7745510935783386
Epoch 6, Loss: 0.7059294581413269
Epoch 7, Loss: 0.6355661153793335
Epoch 8, Loss: 0.579180121421814
Epoch 9, Loss: 0.5462357997894287
Epoch 10, Loss: 0.5308310389518738
Epoch 11, Loss: 0.5219612717628479
Epoch 12, Loss: 0.5164632797241211
Epoch 13, Loss: 0.5150585174560547
Epoch 14, Loss: 0.5155951976776123
Epoch 15, Loss: 0.5139747858047485
Epoch 16, Loss: 0.5080417990684509
Epoch 17, Loss: 0.4979343116283417
Epoch 18, Loss: 0.4848780333995819
Epoch 19, Loss: 0.470370352268219
Epoch 20, Loss: 0.4557682275772095
Epoch 21, Loss: 0.4420378506183624
Epoch 22, Loss: 0.42963576316833496
Epoch 23, Loss: 0.41853103041648865
Epoch 24, Loss: 0.4083535373210907
Epoch 25, Loss: 0.3985302746295929
Epoch 26, Loss: 0.38841602206230164
Epoch 27, Loss: 0.3774488568305969
Epoch 28, Loss: 0.3652629256248474
Epoch 29, Loss: 0.3517509698

In [11]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_lstm.pth')